# Dự đoán trên test và chuẩn bị submission

Notebook chạy trên Kaggle với model cuối E2: SegFormer-B0 + CE.
Nạp `best.pt`, dự đoán đủ 693 ảnh test và tạo ZIP theo định dạng semantic segmentation.
Không huấn luyện lại hoặc cập nhật trọng số.

Thêm ba Input: code mới nhất, kết quả train E2 và ảnh test PhenoBench.
Bật GPU và Internet, chạy các cell từ trên xuống.
Nhãn test không công khai; điểm IoU/mIoU do server PhenoBench tính sau khi nộp.


## 1. Cấu hình

Giữ `MODEL='segformer'`, `LOSS='ce'` cho E2 đã chọn từ validation.
`CHECKPOINT_PATH` trỏ tới `best.pt`, `TEST_IMAGES` trỏ tới thư mục `test/images`.
Để `None` để tự tìm trong Input; khi có nhiều kết quả, điền đường dẫn chính xác.
Thư mục checkpoint cần có `config.yaml` tương ứng.

`RUN_TAG` dùng để tìm checkpoint tự động. `RUN_PREVIEW=True` kiểm tra trước hai ảnh,
sau đó notebook chạy dự đoán toàn bộ test ở bước 5.


In [ ]:
MODEL = 'segformer'
LOSS = 'ce'
RUN_TAG = 'plan20_original_loop_v1'
CHECKPOINT_PATH = None
TEST_IMAGES = None
RUN_PREVIEW = True
RUN_OFFICIAL_VALIDATOR = True


## 2. Chuẩn bị code và thư viện

Code Input có thể là thư mục đã giải nén hoặc `DO_AN_AI_cloud.zip`.
Notebook dùng PyTorch CUDA có sẵn trên Kaggle và cài `requirements-cloud.txt`.


In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

BASE = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
if not INPUT.is_dir():
    raise RuntimeError('Notebook này chạy trên Kaggle.')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Bật GPU trong Settings > Accelerator, khởi động lại phiên rồi chạy từ đầu.')
subprocess.run(['nvidia-smi'], check=True)


def extract_inside(archive_path, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if not (target / member.filename).resolve().is_relative_to(target.resolve()):
                raise ValueError('Đường dẫn không hợp lệ trong ZIP: ' + member.filename)
        archive.extractall(target)


PROJECT = BASE / 'DO_AN_AI'
if not (PROJECT / 'scripts/package_submission.py').is_file():
    roots = sorted({p.parents[1] for p in INPUT.rglob('package_submission.py')
                    if p.parent.name == 'scripts'
                    and (p.parents[1] / 'requirements-cloud.txt').is_file()})
    archives = sorted(INPUT.rglob('DO_AN_AI_cloud.zip'))
    if PROJECT.exists():
        raise RuntimeError('Code trong Working chưa có package_submission.py. Dùng phiên mới và code ZIP mới nhất.')
    if len(roots) == 1:
        shutil.copytree(roots[0], PROJECT)
    elif not roots and len(archives) == 1:
        extract_inside(archives[0], BASE)
    else:
        raise RuntimeError('Thêm đúng một bộ code mới nhất vào Input.')
assert (PROJECT / 'scripts/package_submission.py').is_file(), 'Cập nhật code Input mới nhất.'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(PROJECT / 'requirements-cloud.txt')], check=True)
subprocess.run([sys.executable, '-c',
                'import torch; assert torch.cuda.is_available(); print(torch.__version__); print(torch.cuda.get_device_name(0))'],
               check=True)
print('Project:', PROJECT)


## 3. Tìm ảnh test

`PhenoBench_test.zip` gồm `PhenoBench/test/images/*.png`.
Nếu Input đã giải nén, notebook đọc trực tiếp. Nếu còn ZIP, giải nén vào Working.
Tên file prediction phải giữ nguyên tên ảnh test; chỉ nhận đủ 693 ảnh.


In [ ]:
def find_test_images(search_root):
    candidates = sorted(p for p in search_root.rglob('images')
                        if p.is_dir() and p.parent.name == 'test'
                        and any(p.glob('*.png')))
    if len(candidates) > 1:
        raise RuntimeError('Có nhiều thư mục test/images; điền TEST_IMAGES ở bước 1: ' + str(candidates))
    return candidates[0] if candidates else None


if TEST_IMAGES is None:
    TEST_IMAGES = find_test_images(INPUT)
    if TEST_IMAGES is None:
        target = BASE / 'phenobench_test'
        TEST_IMAGES = find_test_images(target) if target.is_dir() else None
        if TEST_IMAGES is None:
            archives = sorted(INPUT.rglob('PhenoBench_test.zip'))
            if len(archives) != 1:
                raise FileNotFoundError('Thêm PhenoBench_test.zip hoặc test/images đã giải nén vào Input.')
            extract_inside(archives[0], target)
            TEST_IMAGES = find_test_images(target)
if TEST_IMAGES is None:
    raise FileNotFoundError('Không tìm thấy test/images.')
TEST_IMAGES = Path(TEST_IMAGES)
test_files = sorted(TEST_IMAGES.glob('*.png'))
assert TEST_IMAGES.name == 'images' and TEST_IMAGES.parent.name == 'test', TEST_IMAGES
assert len(test_files) == 693, f'Cần đủ 693 ảnh test, tìm thấy {len(test_files)}.'
print('Test images:', TEST_IMAGES)
print('Số ảnh:', len(test_files))


## 4. Kiểm tra checkpoint cuối

Notebook tìm checkpoint theo model, loss và tag trong `config.yaml` cạnh `best.pt`.
Nếu nhiều bản cùng khớp, chỉ định `CHECKPOINT_PATH` ở bước 1.
Cấu hình sidecar cũng được truyền vào `predict.py` để đối chiếu metadata trong checkpoint.


In [ ]:
import yaml

if MODEL != 'segformer' or LOSS != 'ce':
    raise ValueError('Notebook này dành cho model cuối E2: segformer + ce.')
if CHECKPOINT_PATH is None:
    matches = []
    for path in INPUT.rglob('best.pt'):
        config_path = path.parent / 'config.yaml'
        if not config_path.is_file():
            continue
        saved = yaml.safe_load(config_path.read_text(encoding='utf-8')) or {}
        run_path = saved.get('training', {}).get('output_dir', '').replace(chr(92), '/')
        if (saved.get('model_name') == MODEL and saved.get('loss', {}).get('name') == LOSS
                and RUN_TAG in run_path.split('/') and not saved.get('smoke_test')):
            matches.append(path)
    if len(matches) != 1:
        raise RuntimeError('Điền CHECKPOINT_PATH ở bước 1. Các bản tìm được: ' + str(matches))
    CHECKPOINT_PATH = matches[0]
CHECKPOINT_PATH = Path(CHECKPOINT_PATH)
CONFIG_PATH = CHECKPOINT_PATH.parent / 'config.yaml'
if not CHECKPOINT_PATH.is_file() or not CONFIG_PATH.is_file():
    raise FileNotFoundError('Cần best.pt và config.yaml trong cùng thư mục.')
saved = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
if (saved.get('model_name') != MODEL or saved.get('loss', {}).get('name') != LOSS
        or saved.get('smoke_test')):
    raise ValueError('Checkpoint/config không phải E2 SegFormer CE đã train đầy đủ.')

OUTPUT = BASE / 'test_predictions' / 'E2'
PREVIEW_OUTPUT = BASE / 'test_predictions' / 'E2_preview'
print('Checkpoint:', CHECKPOINT_PATH)
print('Config:', CONFIG_PATH)
print('Output:', OUTPUT)


## 5. Dự đoán

Full-image inference, batch 1 và AMP như evaluation trên validation.
Preview dùng hai ảnh, lưu riêng trong `E2_preview`; bước nộp server chỉ nhận đủ 693 ảnh.
Mỗi output phải mới hoặc rỗng. Khi chạy lại thành công, dùng lại kết quả hiện có
và chuyển tới bước 6 thay vì chạy đè.


In [ ]:
def run_prediction(output, limit=None):
    command = [
        sys.executable, '-u', str(PROJECT / 'scripts/predict.py'),
        '--checkpoint', str(CHECKPOINT_PATH), '--config', str(CONFIG_PATH),
        '--model-name', MODEL, '--input-dir', str(TEST_IMAGES),
        '--output-dir', str(output), '--device', 'cuda', '--inference-mode', 'full',
        '--max-figures', '10',
    ]
    if limit is not None:
        command.extend(['--limit', str(limit)])
    subprocess.run(command, cwd=PROJECT, check=True)


if RUN_PREVIEW:
    run_prediction(PREVIEW_OUTPUT, limit=2)


In [ ]:
run_prediction(OUTPUT)


## 6. Kiểm tra và đóng gói

`package_submission.py` kiểm tra đủ 693 tên file, kích thước 1024 × 1024,
mask một kênh với ID 0/1/2; chuyển sang PNG 16-bit và đóng gói:

```text
submission_E2_segformer_ce.zip
  semantics/
    <tên ảnh test>.png
```

ZIP submission chỉ chứa nhãn semantic. `test_predictions_E2.zip` giữ kết quả để xem
và bàn giao, gồm prediction, spray mask, hình minh họa và manifest.
Nếu bật `RUN_OFFICIAL_VALIDATOR`, notebook kiểm tra ZIP bằng devkit PhenoBench.
Validator xác nhận định dạng, chưa tính điểm test.


In [ ]:
SUBMISSION_ZIP = BASE / 'submission_E2_segformer_ce.zip'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/package_submission.py'),
    '--predictions', str(OUTPUT / 'predictions'), '--test-images', str(TEST_IMAGES),
    '--output', str(SUBMISSION_ZIP),
], cwd=PROJECT, check=True)

if RUN_OFFICIAL_VALIDATOR:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'phenobench'], check=True)
    subprocess.run([
        sys.executable, '-m', 'phenobench.tools.validator', '--task', 'semantics',
        '--phenobench_dir', str(TEST_IMAGES.parent.parent), '--zipfile', str(SUBMISSION_ZIP),
    ], check=True)

demo_zip = shutil.make_archive(str(BASE / 'test_predictions_E2'), 'zip',
                               root_dir=OUTPUT.parent, base_dir=OUTPUT.name)
print('Nộp server:', SUBMISSION_ZIP)
print('Kết quả để xem/bàn giao:', demo_zip)
print('Tải từ bảng Output: /kaggle/working > dấu ba chấm cạnh file > Download.')


## 7. Nộp server và lưu điểm

Mở [PhenoBench Benchmarks](https://www.phenobench.org/benchmarks.html), chọn
Semantic segmentation of plants để đi tới Codabench. Đăng nhập, đọc yêu cầu hiện tại
và đăng ký tham gia; nếu cần duyệt tài khoản thì chờ ban tổ chức xác nhận.

Trong My Submissions, chọn phase chấm test và tải `submission_E2_segformer_ce.zip`.
Đối chiếu cấu trúc ZIP với hướng dẫn đang hiển thị trước khi nộp.
Theo dõi trạng thái và log; sau khi chấm xong, xem điểm ở submission hoặc Results.

Lưu submission ID, checkpoint SHA-256 từ `predictions_manifest.json`, cấu hình,
điểm mIoU/IoU từng lớp và ngày nộp vào báo cáo. Điểm test có thể khác validation.
Không đổi model hoặc tham số liên tục theo điểm test.

Server semantic segmentation công bố IoU/mIoU. Weed Coverage và Crop Wrongly
Sprayed Rate của dự án vẫn báo cáo trên validation có GT, trừ khi server trả thêm
thông tin đủ để tính các chỉ số đó.

Tài liệu tham khảo: [validator chính thức](https://github.com/PRBonn/phenobench/blob/main/src/phenobench/tools/validator.py),
[định dạng ảnh PhenoBench](https://www.phenobench.org/dataset.html),
[hướng dẫn Codabench](https://docs.codabench.org/latest/Participants/User_Participating-in-a-Competition/).
